# Nemotron Reasoning Challenge — Colab A100 v2 (improved)

Targets **Kaggle score > 0.61** on top of the previous adapter by:

1. **Wider LoRA targets** — hit attention + MoE FFN + Mamba projections (prev regex only hit Mamba + MoE partial).
2. **RSLoRA + alpha=64** — proper LoRA scale (prev had alpha=32 with r=32 → scale 1.0, too small).
3. **Completion-only loss** — loss computed only on the assistant reply, not the prompt.
4. **NEFTune noise** (α=5) — standard +1–2pp SFT boost.
5. **Packing on** — more gradient signal per step, no wasted pad tokens.
6. **3 epochs** — 1 epoch on 1.6k synthetic was heavy undertraining.
7. **API CoT on real train.csv** — turns on GPT-4o teacher to produce verified reasoning traces for the actual competition distribution. Synthetic alone (prev run) ≠ test distribution.
8. **Balanced per-type caps** — 2000/type instead of 400/type, blended with API-verified real data.

**Runtime:** A100 40GB. QLoRA 4-bit required (30B bf16 = 60GB). Expect ~2–4 h training.

Set OpenAI or Anthropic key before Phase 2 for best results.

## Setup

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files

WORK_ROOT = Path("/content/project").resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Upload udacity_upload.zip (project code)...")
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(WORK_ROOT)
        print(f"Extracted {name} to {WORK_ROOT}")

data_dir = WORK_ROOT / "data"
(data_dir / "reports").mkdir(parents=True, exist_ok=True)
(data_dir / "synthetic").mkdir(parents=True, exist_ok=True)

if not (data_dir / "train.csv").is_file():
    print("Upload train.csv...")
    up2 = files.upload()
    for n in up2:
        Path(n).rename(data_dir / n)

os.chdir(WORK_ROOT)
sys.path.insert(0, str(WORK_ROOT))
assert (WORK_ROOT / "scripts" / "01_eda.py").is_file(), "scripts/ missing"
assert (data_dir / "train.csv").is_file(), "train.csv missing"
print("cwd:", os.getcwd(), " train.csv:", (data_dir / "train.csv").stat().st_size, "bytes")

## Config

In [ ]:
import torch, getpass

MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
HF_TOKEN = os.environ.get("HF_TOKEN", "") or getpass.getpass("HF token (or blank if none): ")

# Teacher-CoT backend for real train.csv (strongly recommended vs synthetic-only).
# Leave empty to SKIP teacher CoT and use synthetic-only (same as prev run).
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "") or getpass.getpass("OpenAI key (blank=synthetic-only): ")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
SKIP_COT = not OPENAI_API_KEY
COT_BACKEND = "openai"
COT_MODEL = "gpt-4o-mini"  # cheaper + fast; swap to gpt-4o if budget allows

# A100 40GB
TRAIN_MAX_SEQ = 4096
TRAIN_BATCH = 2
GRAD_ACCUM = 8        # effective batch = 16
NUM_EPOCHS = 3.0      # was 1.0; small adapter needs more passes
LR = 2e-4             # was 1e-4; standard QLoRA LR

# LoRA
LORA_R = 32           # max per competition rules
LORA_ALPHA = 64       # 2x rank with rsLoRA => stable scale ≈ 11.3
LORA_DROPOUT = 0.05
LORA_TARGET_MODE = "kaggle_nemotron"  # wider regex incl. q/k/v/o/gate_proj (see patched 03_train_lora.py)

# Data
SYNTHETIC_PER_KIND = 600
MAX_PER_TYPE = 2000
LIMIT_TRAIN_CSV_FOR_COT = 0  # 0=all rows from train.csv; set smaller if budget is tight

print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
print(f"max_seq={TRAIN_MAX_SEQ} bs={TRAIN_BATCH}x{GRAD_ACCUM} epochs={NUM_EPOCHS} lr={LR}")
print(f"LoRA r={LORA_R} α={LORA_ALPHA} dropout={LORA_DROPOUT} mode={LORA_TARGET_MODE}")
print(f"Teacher CoT: {'OFF (synthetic only)' if SKIP_COT else f'{COT_BACKEND}/{COT_MODEL}'}")

## Install dependencies (torch/transformers/trl/peft + mamba-ssm + causal-conv1d)

In [ ]:
import subprocess, sys, os, re

def pip_install(*args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
def pip_try(*args):
    try: pip_install(*args); return True
    except subprocess.CalledProcessError: return False

pip_install("-U", "pip", "setuptools", "wheel")
pip_install(
    "transformers>=4.45,<5",
    "peft>=0.12",
    "trl>=0.12",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "psutil", "pandas", "numpy", "scikit-learn", "tqdm",
    "huggingface_hub", "ninja",
)

import torch
_torch_full = torch.__version__
_torch_mm = re.match(r"(\d+\.\d+)", _torch_full).group(1)
_torch_minor = int(_torch_mm.split(".")[1])
_cu = "cu12" if "cu12" in _torch_full else "cu11"
_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
_abi_order = ["cxx11abiTRUE", "cxx11abiFALSE"] if _torch_minor >= 7 else ["cxx11abiFALSE", "cxx11abiTRUE"]

_GH_C = "https://github.com/Dao-AILab/causal-conv1d/releases/download"
_GH_M = "https://github.com/state-spaces/mamba/releases/download"
for pkg, versions in [
    ("causal-conv1d", [("v1.6.1", "causal_conv1d", "1.6.1"), ("v1.5.4", "causal_conv1d", "1.5.4")]),
    ("mamba-ssm",     [("v2.3.1", "mamba_ssm", "2.3.1"), ("v2.2.4", "mamba_ssm", "2.2.4")]),
]:
    gh = _GH_C if "causal" in pkg else _GH_M
    ok = False
    for tag, wn, wv in versions:
        if ok: break
        for abi in _abi_order:
            url = f"{gh}/{tag}/{wn}-{wv}+{_cu}torch{_torch_mm}{abi}-{_py}-{_py}-linux_x86_64.whl"
            if pip_try(url):
                print(f"OK {pkg} {wv} {abi}"); ok = True; break
    if not ok:
        os.environ["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"
        os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
        pip_install("--no-build-isolation", "--no-deps", pkg)

pip_try("--prefer-binary", "nvidia-cutlass-dsl>=4.4", "nvidia-cutlass-dsl-libs-base>=4.4")

for p in ("torch", "transformers", "peft", "trl", "bitsandbytes", "mamba_ssm"):
    try: print(f"  {p}: {__import__(p).__version__}")
    except: print(f"  {p}: MISSING")

## Download base model

In [ ]:
from huggingface_hub import login, snapshot_download
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
MODEL_PATH_LOCAL = snapshot_download(MODEL_ID, resume_download=True)
print("Model cached at:", MODEL_PATH_LOCAL)

## Auto-patch Nemotron modeling for 4-bit compatibility

In [ ]:
import glob, shutil

mfs = set(glob.glob("/root/.cache/huggingface/hub/**/modeling_nemotron_h.py", recursive=True)
         + glob.glob("/root/.cache/huggingface/modules/**/modeling_nemotron_h.py", recursive=True))
for mf in mfs:
    lines = Path(mf).read_text().splitlines(True)
    changed = False
    for i, line in enumerate(lines):
        if "final_hidden_states.index_add_(0, token_indices, weighted_output)" in line and "weighted_output.to(" not in line:
            lines[i] = line.replace("weighted_output)", "weighted_output.to(final_hidden_states.dtype))")
            changed = True
        if ".to(expert_dtype)" in line:
            lines[i] = line.replace(".to(expert_dtype)", ".to(torch.bfloat16)"); changed = True
    if changed:
        Path(mf).write_text("".join(lines))
        pc = os.path.dirname(mf) + "/__pycache__"
        if os.path.exists(pc): shutil.rmtree(pc)
        print("patched:", mf)
    else:
        print("ok:", mf)

## Phase 1 — EDA

In [ ]:
subprocess.run([sys.executable, "scripts/01_eda.py",
                "--data-dir", "data", "--report-dir", "data/reports",
                "--tokenizer-model", str(MODEL_PATH_LOCAL)], check=True)

## Phase 2 — Build SFT data (synthetic + optional verified teacher CoT on real train.csv)

In [ ]:
cot_args = ["--skip-cot"] if SKIP_COT else [
    "--cot-backend", COT_BACKEND, "--cot-model", COT_MODEL,
    "--cot-max-tokens", "4096",
]
if not SKIP_COT and LIMIT_TRAIN_CSV_FOR_COT > 0:
    cot_args += ["--limit-train", str(LIMIT_TRAIN_CSV_FOR_COT)]

cmd = [sys.executable, "scripts/02_prepare_data.py",
       "--data-dir", "data", "--synthetic-dir", "data/synthetic",
       "--output", "data/train_sft.jsonl",
       "--tokenizer-model", str(MODEL_PATH_LOCAL),
       "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
       "--max-per-type", str(MAX_PER_TYPE)] + cot_args
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Phase 3 — LoRA SFT (wider targets + rsLoRA + packing + completion-only + NEFTune)

In [ ]:
import gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
train_env = os.environ.copy()
train_env["TOKENIZERS_PARALLELISM"] = "false"
train_env["NEMOTRON_KAGGLE_PATCHES"] = "0"
train_env["PYTHONUNBUFFERED"] = "1"

cmd = [sys.executable, "scripts/03_train_lora.py",
       "--data-path", "data/train_sft.jsonl",
       "--output-dir", "lora_adapter",
       "--checkpoint-dir", "lora_output",
       "--model-path", str(MODEL_PATH_LOCAL),
       "--lora-target-mode", LORA_TARGET_MODE,
       "--lora-r", str(LORA_R),
       "--lora-alpha", str(LORA_ALPHA),
       "--lora-dropout", str(LORA_DROPOUT),
       "--batch-size", str(TRAIN_BATCH),
       "--grad-accum", str(GRAD_ACCUM),
       "--epochs", str(NUM_EPOCHS),
       "--lr", str(LR),
       "--max-seq-length", str(TRAIN_MAX_SEQ),
       "--warmup-ratio", "0.05",
       "--max-grad-norm", "1.0",
       "--neftune-alpha", "5.0",
       "--packing",
       # --completion-only and --packing are mutually exclusive in trl; prefer packing on A100.
       "--force-peft",
       "--no-nemotron-kaggle-patches",
       "--dataloader-workers", "0"]
gc.collect()
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, env=train_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end="", flush=True)
rc = proc.wait()
if rc != 0: raise subprocess.CalledProcessError(rc, cmd)

## Phase 4 — Package submission

In [ ]:
subprocess.run([sys.executable, "scripts/05_package_submission.py",
                "--adapter-dir", "lora_adapter",
                "--output", "submission.zip"], check=True)
zp = WORK_ROOT / "submission.zip"
print("submission.zip:", zp.is_file(), zp.stat().st_size if zp.is_file() else 0, "bytes")

## Download + submit

In [ ]:
from google.colab import files
files.download(str(WORK_ROOT / "submission.zip"))